#  Forward pass

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann
* https://pytorch.org/docs/stable/generated/torch.matmul.html
* https://machinelearningmastery.com/choose-an-activation-function-for-deep-learning/
* https://machinelearningmastery.com/loss-and-loss-functions-for-training-deep-learning-neural-networks/
* https://kidger.site/thoughts/jaxtyping/
* https://github.com/patrick-kidger/torchtyping/tree/master

## Задачи для совместного разбора

In [150]:
pip install torchtyping

In [151]:
from torchtyping import TensorType, patch_typeguard
from typeguard import typechecked
import torch as th

Scalar = TensorType[()]
patch_typeguard()

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте нейрон с заданными весами `weights` и `bias`. Пропустите вектор `inputs` через нейрон и выведите результат.

In [152]:
class Neuron:
    def __init__(self, n_features: int, bias: float):
        # <создать атрибуты объекта weights и bias>
        self.weights: TensorType[n_features] = th.randn(n_features, requires_grad=False) #Количество весов зависит от входимых признаков
        self.bias: float = bias #Смещение

    def forward(self, inputs: TensorType["n_features"]) -> Scalar:
        return th.dot(self.weights, inputs) + self.bias #Матричное умножение с учетом смещения на self.bias

    @property
    def w(self):
      return self.weights

In [153]:
inputs = th.tensor([4.5, 2, 1.6, -3]) #Вводные признаки
neuron1 = Neuron(4, 3.4) #создание объекта класса Нейрон
print(neuron1.forward(inputs))
print(neuron1.w)

tensor(3.9919)
tensor([ 0.5539,  1.5386, -1.9591,  0.6144])


2\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ReLU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/f4353f4e3e484130504049599d2e7b040793e1eb)

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

In [154]:
class ReLU:
    # @typechecked
    def forward(self, inputs: TensorType["n_features"]) -> TensorType["n_features"]:
      '''inputs[inputs < 0] = 0
      return inputs'''
      return th.relu(inputs)

In [155]:
func = ReLU()
n_features = th.randn(4, 3, dtype=th.float32)
Y = func.forward(n_features)
n_features, Y

(tensor([[-0.4215, -0.5258,  0.6526],
         [-1.9206,  2.2693,  0.7444],
         [ 0.5364, -0.1006,  2.9481],
         [-1.1660,  1.7000, -0.3135]]),
 tensor([[0.0000, 0.0000, 0.6526],
         [0.0000, 2.2693, 0.7444],
         [0.5364, 0.0000, 2.9481],
         [0.0000, 1.7000, 0.0000]]))

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь MSE:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/e258221518869aa1c6561bb75b99476c4734108e)
где $Y_i$ - правильный ответ для примера $i$, $\hat{Y_i}$ - предсказание модели для примера $i$, $n$ - количество примеров в батче.

In [156]:
class MSELoss:
    @typechecked
    def forward(self, y_pred: TensorType["batch"], y_true: TensorType["batch"]) -> Scalar:
        return th.mean(th.square(y_true - y_pred)) # <реализовать логику MSE>

In [157]:
y_pred = th.tensor([1.0, 3.0, 5.0])
y_true = th.tensor([2.0, 3.0, 4.0])

In [158]:
loss = MSELoss()
loss.forward(y_pred, y_true)

tensor(0.6667)

## Задачи для самостоятельного решения

### Cоздание полносвязных слоев

<p class="task" id="1"></p>

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте полносвязный слой из `n_neurons` нейронов с `n_features` весами у каждого нейрона (инициализируются из стандартного нормального распределения) и опциональным вектором смещения.

$$y = xW^T + b$$

Пропустите вектор `inputs` через слой и выведите результат. Результатом прогона сквозь слой должна быть матрица размера `batch_size` x `n_neurons`.

- [ ] Проверено на семинаре

In [159]:
class Linear:
    def __init__(self, n_neurons: int, n_features: int, bias: bool = False) -> None:
        self.weights = th.randn(n_neurons, n_features) #Веса нейроны х признаки
        if bias:
          self.bias = th.zeros(n_neurons) #Общее смещение
        else:
          self.bias = None

    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "n_neurons"]:
        if self.bias:
          Y = inputs @ self.weights.T + self.bias #Матричное произведение Исходных данных (batch X feats) и весов слоя Модели.T (feats X neurs)
        else:
          Y = inputs @ self.weights.T
        return Y #На выходе (batch X neurs)

In [160]:
n_features, n_neurons, n_samples = 4, 3, 8
layer1 = Linear(n_neurons, n_features) #Создаем первый слой

inputs = th.randn(n_samples, n_features, dtype=th.float32) #Создаем случайную матрицу (batch X feats)
Y = layer1.forward(inputs) #Пропускаем через слой нейронной сети
Y, Y.shape

(tensor([[-2.1467, -1.3766, -2.6089],
         [ 0.5305, -0.8226, -0.0242],
         [-1.8175,  0.2680,  1.8419],
         [ 4.5789,  1.3067,  1.6658],
         [-2.2219, -1.2110,  0.3136],
         [ 0.7138, -0.3662, -0.6784],
         [-0.1831,  0.9834,  2.3105],
         [-1.5218, -0.2099,  2.3388]]),
 torch.Size([8, 3]))

<p class="task" id="2"></p>

2\. Используя решение предыдущей задачи, создайте 2 полносвязных слоя и пропустите тензор `inputs` последовательно через эти два слоя. Количество нейронов в первом слое выберите произвольно, количество нейронов во втором слое выберите так, чтобы результатом прогона являлась матрица `batch_size x 7`.

- [ ] Проверено на семинаре

In [161]:
layer1 = Linear(3, 4)  #\ Инициализация двух слоев нейронной сети
layer2 = Linear(7, 3)  #/

inputs = th.randn(15, 4, dtype=th.float32)

Y = layer1.forward(inputs) #\ Исходные данные прошли через 2 слоя сети
Y = layer2.forward(Y)      #/

print(Y, Y.shape) #На выходе (batch X 7)

tensor([[-0.8903,  0.6083,  1.7981, -0.5030, -0.6441, -3.1815,  4.9742],
        [-0.4435,  5.9136,  1.0091,  1.7065, -0.4606,  4.5199, -2.6113],
        [-1.1014, -3.5082, -3.7974,  0.2529,  1.8010, -1.1290, -3.2458],
        [-0.4941,  0.0940, -0.4167,  0.1967,  0.2404, -0.2631, -0.1686],
        [ 0.3337, -3.8323, -1.6705, -0.7013,  0.7190, -1.5711, -0.6592],
        [ 0.5259, -1.4216,  0.3021, -0.6247, -0.1748, -1.0141,  1.1219],
        [-1.2508,  4.7085,  1.6053,  1.0350, -0.5816,  0.9760,  1.2606],
        [-0.8953,  1.3123, -0.3723,  0.6117,  0.2561,  0.3149, -0.5378],
        [ 0.4111,  2.2875,  2.1587, -0.0403, -0.9995,  0.5809,  1.9367],
        [-1.5972,  0.1645,  2.0917, -0.7820, -0.6643, -5.3176,  7.2721],
        [ 1.1684, -0.7986,  0.5961, -0.5159, -0.3955,  0.4685,  0.0778],
        [ 0.2360, -3.3898, -2.1797, -0.3413,  0.9387, -0.6159, -2.0500],
        [ 2.5457, -1.5823, -0.2318, -0.4589, -0.2218,  3.1158, -3.4013],
        [ 0.9201,  3.4652,  0.4431,  1.0652, -0.367

### Создание функций активации

<p class="task" id="3"></p>

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации softmax:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/6d7500d980c313da83e4117da701bf7c8f1982f5)

$$\overrightarrow{x} = (x_1, ..., x_J)$$

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации. Строки матрицы трактовать как выходы линейного слоя некоторого классификатора для 4 различных примеров. Функция должна применяться переданной на вход матрице построчно.

- [ ] Проверено на семинаре

In [162]:
class Softmax:
    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "feats"]:
        exp = th.exp(inputs)
        return exp / exp.sum(dim=1, keepdim=True) #Постолбчатая экспонента коэф-тов деленная на экспоненту суммы коэф-тов строки

In [163]:
smax = Softmax()
X = th.randn(4, 3, dtype=th.float32)
print(f'Исходная матрица:\n {X}\nМатрица после сигмоиды:\n {smax.forward(X)}')

Исходная матрица:
 tensor([[ 0.2471, -0.7689,  0.3265],
        [-1.8493, -0.3414, -0.9755],
        [ 1.2427, -0.7051, -0.9155],
        [ 0.5090, -0.5790, -1.2657]])
Матрица после сигмоиды:
 tensor([[0.4090, 0.1481, 0.4429],
        [0.1264, 0.5709, 0.3028],
        [0.7948, 0.1133, 0.0918],
        [0.6638, 0.2236, 0.1125]])


<p class="task" id="4"></p>

4 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ELU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/eb23becd37c3602c4838e53f532163279192e4fd)

Создайте матрицу размера 4x3, заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

- [ ] Проверено на семинаре

In [164]:
class ELU:
    def __init__(self, alpha: float) -> None:
        self.alpha = alpha

    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "feats"]:
        return th.where( #Метод where, условие -> функция при выполении условия -> функция при не выполнении условия
            inputs < 0,
            self.alpha * (th.exp(inputs) - 1),
            inputs
        )

In [165]:
elu = ELU(0.95)
X = th.randn(4, 3, dtype=th.float32)
Y = elu.forward(X)
print(f'Исходная матрица:\n {X}\nМатрица после ELU:\n {Y}')

Исходная матрица:
 tensor([[ 0.4521, -0.8823,  0.3214],
        [-0.1889, -0.4091, -0.5889],
        [ 1.0405, -0.9942, -0.6183],
        [-1.1015, -0.1725, -0.3306]])
Матрица после ELU:
 tensor([[ 0.4521, -0.5569,  0.3214],
        [-0.1635, -0.3189, -0.4228],
        [ 1.0405, -0.5985, -0.4381],
        [-0.6342, -0.1506, -0.2675]])


### Создание функции потерь

<p class="task" id="5"></p>

5 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь CrossEntropyLoss:

$$y_i = (y_{i,1},...,y_{i,k})$$

<img src="https://i.ibb.co/93gy1dN/Screenshot-9.png" width="200">

$$ CrossEntropyLoss = \frac{1}{n}\sum_{i=1}^{n}{L_i}$$
где $y_i$ - вектор правильных ответов для примера $i$, $\hat{y_i}$ - вектор предсказаний модели для примера $i$; $k$ - количество классов, $n$ - количество примеров в батче.

Создайте полносвязный слой с 2 нейронами и прогнать через него батч `inputs`. Полученный результат пропустите через функцию активации Softmax. Посчитайте значение функции потерь, трактуя вектор `y` как вектор правильных ответов.

- [ ] Проверено на семинаре

In [166]:
class CrossEntropyLoss:
    def forward(self, y_pred: TensorType["batch", float], y_true: TensorType["batch", int]):
        CEL = th.mean(- th.sum(y_true * th.log(y_pred))) #Реализовал формулу выше
        return CEL

In [167]:
CEL = CrossEntropyLoss()
Y_l1 = layer1.forward(inputs)
Y_l2 = layer2.forward(Y_l1)

Y_smax = smax.forward(Y_l2)
print(f'{inputs}\n{Y_smax}')

tensor([[-0.6411, -0.1848,  0.0505, -1.0357],
        [ 0.5543,  0.5863, -2.2538, -0.7633],
        [-0.3789,  1.3968,  1.0666,  1.5744],
        [-0.2878, -0.1394, -0.0568,  0.3477],
        [ 0.3350,  1.6459,  1.3684,  0.0565],
        [-0.3202, -1.4887,  0.5503,  0.5589],
        [-0.4191, -0.4649, -1.6391, -0.6198],
        [-0.6694, -1.1404, -0.5421,  0.9014],
        [ 0.1772, -0.6135, -0.7060, -0.9819],
        [-0.9146,  0.5684,  0.3438, -1.6449],
        [ 0.5610,  0.0581,  0.3148, -0.4448],
        [-0.1096, -0.0058,  1.0920,  1.2906],
        [ 1.3336,  0.0397,  0.4197,  0.0689],
        [ 0.7865, -0.4950, -1.4518,  0.0982],
        [ 0.4367, -0.1771,  0.1942,  0.2949]])
tensor([[2.6643e-03, 1.1924e-02, 3.9186e-02, 3.9247e-03, 3.4080e-03, 2.6949e-04,
         9.3862e-01],
        [1.3613e-03, 7.8487e-01, 5.8188e-03, 1.1687e-02, 1.3383e-03, 1.9477e-01,
         1.5577e-04],
        [4.1088e-02, 3.7021e-03, 2.7724e-03, 1.5917e-01, 7.4849e-01, 3.9966e-02,
         4.8126e-03],


In [170]:
noise_level = 0.05
Y_noisy = Y_smax + th.randn_like(Y_smax) * noise_level #Добавление Гаусс-шума с 5% значимостью от Y_true значений (не понял, что за y в условии)
CEL.forward(Y_smax, Y_noisy)

tensor(16.2480)

<p class="task" id="6"></p>

6 Модифицируйте MSE, добавив L2-регуляризацию.

$$MSE_R = MSE + \lambda\sum_{i=1}^{m}w_i^2$$

где $\lambda$ - коэффициент регуляризации; $w_i$ - веса модели.

- [ ] Проверено на семинаре

In [171]:
class MSERegularized:
    def __init__(self, lambda_: float) -> None:
        self.lambda_ = lambda_

    def data_loss(
            self,
            y_pred: TensorType["batch"],
            y_true: TensorType["batch"],
    ) -> Scalar:
        return th.mean(th.square(y_pred - y_true))

    def reg_loss(self, weights: TensorType["batch", 1])  -> Scalar:
        return th.sum(th.square(weights))

    def forward(
        self,
        y_pred: TensorType["batch"],
        y_true: TensorType["batch"],
        weights: TensorType["batch", 1],
    ) -> Scalar:
        return self.data_loss(y_pred, y_true) + self.lambda_ * self.reg_loss(weights)